# 🚀 End-to-End Data Pipeline using Fivetran + DBT Core + Git
### (Production-Level Step-by-Step with Commands & Notes)

---

## 📌 Overview

Modern Data Stack:

- Fivetran → EL (Extract + Load)
- DBT Core → Transform
- Git → Version Control + CI/CD
- Snowflake → Data Warehouse

---

## 🏗️ Architecture

Source Systems (Shopify / CRM / DB)
        ↓
     Fivetran
        ↓
Raw Tables (Snowflake)
        ↓
     DBT Core
        ↓
Analytics Tables (Gold Layer)
        ↓
 BI / Dashboard

---

## 🧠 Use Case

Data Sources:
- Orders
- Customers
- Products

Goal:
- Clean → Transform → Analyze

---

# 🔄 STEP-BY-STEP IMPLEMENTATION

---

## ✅ Step 1: Fivetran Setup

### What Fivetran Does
- Extracts data
- Loads into warehouse
- Handles:
  - Incremental loads
  - Schema changes
  - CDC

---

### Example Raw Tables

raw.shopify.orders
raw.shopify.customers
raw.shopify.products

---

### Fivetran System Columns

_fivetran_synced
_fivetran_deleted
_fivetran_id

---

## ✅ Step 2: Initialize DBT Project

dbt init my_project
cd my_project

---

## ✅ Step 3: Setup Git

git init
git add .
git commit -m "Initial dbt project with fivetran setup"

---

## ✅ Step 4: Configure profiles.yml

my_profile:
  target: dev
  outputs:
    dev:
      type: snowflake
      account: xyz
      user: user
      password: pass
      role: analyst
      database: raw_db
      warehouse: compute_wh
      schema: dbt_dev

---

## ✅ Step 5: Define Sources

version: 2

sources:
  - name: fivetran_raw
    database: raw_db
    schema: shopify

    tables:
      - name: orders
      - name: customers
      - name: products

---

## ✅ Step 6: Staging Layer

SELECT
    id AS order_id,
    customer_id,
    total_price AS amount,
    created_at,
    _fivetran_synced
FROM {{ source('fivetran_raw', 'orders') }}
WHERE _fivetran_deleted = FALSE

---

## ✅ Step 7: Intermediate Layer

SELECT
    order_id,
    customer_id,
    amount,
    CASE
        WHEN amount > 1000 THEN 'high_value'
        ELSE 'normal'
    END AS order_category
FROM {{ ref('stg_orders') }}

---

## ✅ Step 8: Mart Layer

SELECT
    o.order_id,
    c.customer_name,
    p.product_name,
    o.amount,
    o.order_category
FROM {{ ref('int_orders') }} o
JOIN {{ ref('stg_customers') }} c
    ON o.customer_id = c.customer_id
JOIN {{ ref('stg_products') }} p
    ON o.product_id = p.product_id

---

## ✅ Step 9: Run DBT

dbt run

---

## ✅ Step 10: Testing

dbt test

---

## ✅ Step 11: Incremental Model

{{ config(materialized='incremental') }}

SELECT *
FROM {{ ref('stg_orders') }}

{% if is_incremental() %}
WHERE _fivetran_synced > (SELECT MAX(_fivetran_synced) FROM {{ this }})
{% endif %}

---

## ✅ Step 12: Git Workflow

git checkout -b feature/orders-model
git add .
git commit -m "Added orders transformation"
git push origin feature/orders-model

---

## ✅ Step 13: CI/CD

dbt deps
dbt test
dbt run --target prod

---

# 📌 FINAL FLOW

Source → Fivetran → Raw → DBT → Mart → Dashboard

---

# 🎯 KEY TAKEAWAYS

- Fivetran = ingestion
- DBT = transformation
- Git = version control
